In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
# setting up a custom blue-white-orange color map
from matplotlib.colors import LinearSegmentedColormap

colors = np.array([( 37,  52, 148), ( 44, 127, 184), ( 65, 182, 196), (161, 218, 180), (255, 255, 204),
                   (255, 255, 255),
                   (255, 255, 204), (254, 204,  92), (253, 141,  60), (240,  59,  32), (189,   0,  38)]) / 255.

cmc = LinearSegmentedColormap.from_list('BYWYR', colors, N=101)


In [ ]:
def angle_to_l(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 180. / x[~near_zero]
    return x


In [ ]:
def freq_to_time(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 1. / x[~near_zero]
    return x


In [ ]:
def wavenumber_to_wavelength(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 360. / x[~near_zero]
    return x


In [ ]:
# grid
ntrunc=72
nSampWin = 720
nWindow = 59
spd = 2
frequency = np.fft.fftfreq(nSampWin, 1./spd)
ff = frequency[0:int(nSampWin/2)]
ll = np.arange(0, ntrunc+1, 1)
mm = np.arange(-ntrunc, ntrunc+1, 1)

In [ ]:
# estimated parameters
epsilon_0 = 5.8
lambda_0 = 0.060
tau_0 = 2.3


In [ ]:
# comp psd -- l
def psd_l(Fwflm):

    # compute power
    Pwflm = np.abs(Fwflm[:, :, :, :])**2

    # to get psd need to scale (divide) by frequency spacing = spd/(nSampWin)
    Pwflm /= (spd / nSampWin)

    # some power is lost when using hanning window
    Pwflm *= 8. / 3.

    # average over m
    Pwfl_mean = np.zeros((nWindow, int(nSampWin/2), ntrunc+1))
    for l in range(0, ntrunc+1):
        Pwfl_mean[:, :, l] = np.mean(Pwflm[:, :, l, ntrunc-l:ntrunc+l+1], axis=2)

    # sum over m
    Pwfl_sum = np.zeros((nWindow, int(nSampWin/2), ntrunc+1))
    for l in range(0, ntrunc+1):
        Pwfl_sum[:, :, l] = np.sum(Pwflm[:, :, l, ntrunc-l:ntrunc+l+1], axis=2)

    return Pwfl_mean, Pwfl_sum

In [ ]:
# analytic psd -- l
def psd_l_analytic(ff, ll):

    Pfl_analytic = np.zeros((int(nSampWin/2), ntrunc+1))

    for f in range(int(nSampWin/2)):
        for l in range(0, ntrunc+1):
            tau_l = tau_0 / (1 + lambda_0**2 * l * (l+1))
            epsilon_l = epsilon_0 * tau_l / tau_0
            Pfl_analytic[f, l] = 2 * epsilon_l**2 * tau_l / (1 + ff[f]**2 * tau_l**2 * 4 * np.pi**2)

    return Pfl_analytic

In [ ]:
# comp psd -- m
def psd_m(Fwflm):

    # compute power
    Pwflm = np.abs(Fwflm[:, :, :, :])**2

    # to get psd need to scale (divide) by frequency spacing = spd/(nSampWin)
    Pwflm /= (spd / nSampWin)

    # some power is lost when using hanning window
    Pwflm *= 8. / 3.

    # average over l
    Pwfm_mean = np.zeros((nWindow, int(nSampWin/2), 2*ntrunc+1))
    for m in range(-ntrunc, ntrunc+1):
        Pwfm_mean[:, :, ntrunc+m] = np.mean(Pwflm[:, :, np.abs(m):ntrunc+1, ntrunc+m], axis=2)

    # sum over l
    Pwfm_sum = np.zeros((nWindow, int(nSampWin/2), 2*ntrunc+1))
    for m in range(-ntrunc, ntrunc+1):
        Pwfm_sum[:, :, ntrunc+m] = np.sum(Pwflm[:, :, np.abs(m):ntrunc+1, ntrunc+m], axis=2)

    return Pwfm_mean, Pwfm_sum

In [ ]:
# analytic psd -- m
def psd_m_analytic(ff, ll, mm):

    Pflm_analytic = np.zeros((int(nSampWin/2), ntrunc+1, 2*ntrunc+1))

    for f in range(int(nSampWin/2)):
        for l in range(0, ntrunc+1):
            tau_l = tau_0 / (1 + lambda_0**2 * l * (l+1))
            epsilon_l = epsilon_0 * tau_l / tau_0
            Pflm_analytic[f, l, ntrunc-l:ntrunc+l+1] = 2 * epsilon_l**2 * tau_l / (1 + ff[f]**2 * tau_l**2 * 4 * np.pi**2)

    # average over l
    Pfm_analytic = np.zeros((int(nSampWin/2), 2*ntrunc+1))
    for m in range(-ntrunc, ntrunc+1):
        Pfm_analytic[:, ntrunc+m] = np.mean(Pflm_analytic[:, np.abs(m):ntrunc+1, ntrunc+m], axis=1)

    return Pfm_analytic

In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
data_dir = base_dir / "data"
save_dir = base_dir / "figures"

In [ ]:
file_name = "olr-2xdaily-1981-2010-space-time-analysis-window-360-skip-180.nc"
ds_observed = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fwflm_observed = (ds_observed.Fwflm_real.values[:, 0:int(nSampWin/2), :, :] +
                  1j * ds_observed.Fwflm_imag.values[:, 0:int(nSampWin/2), :, :])

In [ ]:
# background realization
file_name = "ou-realization-2024-space-time-analysis-window-360-skip-180-epsilon0-5.8-lambda0-0.06-tau0-2.3.nc"
ds_ou = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fwflm_ou = (ds_ou.Fwflm_real.values[:, 0:int(nSampWin/2), :, :] +
            1j * ds_ou.Fwflm_imag.values[:, 0:int(nSampWin/2), :, :])

In [ ]:
# precomputed p-values
file_name = "boot-statistics-spectral-space-realization-2024-epsilon0-5.8-lambda0-0.06-tau0-2.3.nc"
ds_boot = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
p_value_m = ds_boot.p_value_m.values

In [ ]:
# psd -- m
Pwfm_observed, _ = psd_m(Fwflm_observed)
Pwfm_ou, _ = psd_m(Fwflm_ou)

Pfm_observed = np.mean(Pwfm_observed, axis=0)
Pfm_ou = np.mean(Pwfm_ou, axis=0)


In [ ]:
# foreground
Bfm = Pfm_observed / Pfm_ou


In [ ]:
fig_size = (14.80/2.54, 16.20/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.40, 02.80, 12.00, 12.00])))

cs5 = ax[0].contourf(mm, ff, np.where(p_value_m < 0.001, np.log10(Bfm[:, ::-1]), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')
cs5 = ax[0].contourf(mm, ff, np.where(p_value_m < 0.001, np.log10(Bfm[:, ::-1]), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')

ax[0].set_title(r'Regions of significance', fontsize=10, fontweight='bold')
ax[0].set_xlabel(r'Wavenumber ($m$)', fontsize=10)
ax[0].set_ylabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
ax[0].tick_params(axis='both', which='both', labelleft=True, labelsize=9)
ax[0].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[0].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[0].set_xticks(np.array([-60, -30, 0, 30, 60]))
ax[0].set_xlim(-60, 60)
ax[0].set_ylim(1/360, 1)

secyax = ax[0].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_ylabel('Period [day]', fontsize=10)
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[0].text(00.05, 00.95, 'F', ha='left', va='top', transform=ax[0].transAxes, fontsize=10, fontweight='bold', color='black')

cax = fig.add_axes(ax_pos_cm_to_absolute(fig_size, [2.40, 01.10, 10.00, 00.25])) 
cbar = fig.colorbar(cs5, cax=cax, orientation='horizontal', extend='neither') #, label=r'$\mathrm{W m^{-2}}$')
cax.set_xlabel(r'$\log_{10}$(Ratio)', fontsize=10)
cax.set_xticks(np.array([-0.3, 0.0, 0.3])) #, labels=['-0.6', '-0.3', '0.0', '+0.3', '+0.6'])
cax.tick_params(axis='both', which='both', labelsize=9)

ax[0].grid()


# ---------- render filters ----------

# Satellites
ax[0].add_patch(mpl.patches.Rectangle((11.5, 0.08), 5, 0.06, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[0].add_patch(mpl.patches.Rectangle((-26, 0.87), 20, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[0].add_patch(mpl.patches.Rectangle((-17, 0.76), 5, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[0].add_patch(mpl.patches.Rectangle((-32, 0.76), 5, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))

# kelvin wheeler and kiladis
ax[0].plot(mm[ntrunc+1:ntrunc+7], 0.05 + ( (90*9.8)**0.5 * (mm[ntrunc+1:ntrunc+7]-1) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05 + ( (90*9.8)**0.5 * (mm[ntrunc+6]-1) ) * 86400 / 4e7, xmin = 6, xmax = 14, color='black', linestyle='-', linewidth=1)
ax[0].vlines(x = 14, ymin = 0.05 + ( (8*9.8)**0.5 * (mm[ntrunc+14]-2) ) * 86400 / 4e7, ymax = 0.05 + ( (90*9.8)**0.5 * (mm[ntrunc+6]-1) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].plot(mm[ntrunc+2:ntrunc+15], 0.05 + ( (8*9.8)**0.5 * (mm[ntrunc+2:ntrunc+15]-2) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05, xmin = 1, xmax = 2, color='black', linestyle='-', linewidth=1)

ax[0].text(9, 0.3, '1', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='black')

# eastward baroclinic rossby
y1 = 0.05 + ( (150*9.8)**0.5 * (mm[ntrunc+9]-1) ) * 86400 / 4e7
y2 = 0.05 + ( (8*9.8)**0.5 * (mm[ntrunc+20]-2) ) * 86400 / 4e7
m1 = (y1 - y2) / (9 - 20)
ax[0].plot(mm[ntrunc+1:ntrunc+10], 0.05 + ( (150*9.8)**0.5 * (mm[ntrunc+1:ntrunc+10]-1) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].plot(mm[ntrunc+9:ntrunc+21], y1 + m1 * (mm[ntrunc+9:ntrunc+21]-9), color='black', linestyle='-', linewidth=1)
ax[0].plot(mm[ntrunc+2:ntrunc+21], 0.05 + ( (8*9.8)**0.5 * (mm[ntrunc+2:ntrunc+21]-2) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05, xmin = 1, xmax = 2, color='black', linestyle='-', linewidth=1)

ax[0].text(12, 0.5, '2', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='black')

# westward baroclinic rossby
y1 = 0.05 - ( (150*9.8)**0.5 * (mm[ntrunc-9]+1) ) * 86400 / 4e7
y2 = 0.05 - ( (8*9.8)**0.5 * (mm[ntrunc-20]+2) ) * 86400 / 4e7
m1 = (y1 - y2) / (9 - 20)
ax[0].plot(mm[ntrunc-9:ntrunc], 0.05 - ( (150*9.8)**0.5 * (mm[ntrunc-9:ntrunc]+1) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].plot(mm[ntrunc-20:ntrunc-8], y1 - m1 * (mm[ntrunc-20:ntrunc-8]+9), color='black', linestyle='-', linewidth=1)
ax[0].plot(mm[ntrunc-20:ntrunc-1], 0.05 - ( (8*9.8)**0.5 * (mm[ntrunc-20:ntrunc-1]+2) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05, xmin = -2, xmax = -1, color='black', linestyle='-', linewidth=1)

ax[0].text(-12, 0.5, '3', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='black')

# center lobe
ax[0].plot(mm[ntrunc-4:ntrunc], 0.05 - ( (1400*9.8)**0.5 * (mm[ntrunc-4:ntrunc]+1) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05 + ( (1400*9.8)**0.5 * (mm[ntrunc+4]-1) ) * 86400 / 4e7, xmin = -4, xmax = 4, color='black', linestyle='-', linewidth=1)
ax[0].plot(mm[ntrunc+1:ntrunc+5], 0.05 + ( (1400*9.8)**0.5 * (mm[ntrunc+1:ntrunc+5]-1) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05, xmin = -1, xmax = 1, color='black', linestyle='-', linewidth=1)

ax[0].text(0, 0.7, '4', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='white')

# side lobes
ax[0].plot(mm[ntrunc+10:ntrunc+56], 0.05 + ( (5*9.8)**0.5 * (mm[ntrunc+10:ntrunc+56]-10) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].vlines(x = 55, ymin = 0.05, ymax = 0.05 + ( (5*9.8)**0.5 * (mm[ntrunc+55]-10) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05, xmin = 10, xmax = 55, color='black', linestyle='-', linewidth=1)

ax[0].text(40, 0.3, '5', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='black')

ax[0].plot(mm[ntrunc-55:ntrunc-9], 0.05 - ( (5*9.8)**0.5 * (mm[ntrunc-55:ntrunc-9]+10) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].vlines(x = -55, ymin = 0.05, ymax = 0.05 - ( (5*9.8)**0.5 * (mm[ntrunc-55]+10) ) * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 0.05, xmin = -55, xmax = -10, color='black', linestyle='-', linewidth=1)

ax[0].text(-40, 0.3, '6', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='black')

# mjo 
ax[0].hlines(y = 1./30., xmin = 1, xmax = 6, color='black', linestyle='-', linewidth=1)
ax[0].vlines(x = 1, ymin = 1./96., ymax = 1./30., color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 1./96., xmin = 1, xmax = 6, color='black', linestyle='-', linewidth=1)
ax[0].vlines(x = 6, ymin = 1./96., ymax = 1./30., color='black', linestyle='-', linewidth=1)

ax[0].text(3.5, 1./53, '7', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='black')

# rossby equatorial
ax[0].plot(mm[ntrunc-6:ntrunc], - mm[ntrunc-6:ntrunc] / 4e7 * 86400 / (2*np.pi)  * 2.3e-11 / ( (mm[ntrunc-6:ntrunc] / 4e7)**2 + 2.3e-11 / (90*9.8)**0.5 ), color='black', linestyle='-', linewidth=1)
ax[0].hlines(y = 1./96., xmin = -6, xmax = -1, color='black', linestyle='-', linewidth=1)
ax[0].vlines(x = -6, ymin = 1./96., ymax = - mm[ntrunc-6] / 4e7 * 86400 / (2*np.pi)  * 2.3e-11 / ( (mm[ntrunc-6] / 4e7)**2 + 2.3e-11 / (90*9.8)**0.5 ), color='black', linestyle='-', linewidth=1)

ax[0].text(-5., 1./43., '8', ha='center', va='center', transform=ax[0].transData, fontsize=10, fontweight='regular', color='black')


In [ ]:
file_name = "fig-s05"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')